# TP partie 2 : comptage de mots avec Apache Spark (PySpark)

On refait le comptage d'occurrences de mots avec Spark, en utilisant PySpark, puis on le compare à la version Python simple sur les trois mêmes fichiers que dans la partie 1.

## 1. Installation

PySpark s'installe avec pip et contient déjà Spark. Spark 4 a besoin de Java 17 (Hadoop utilisait Java 11), on le télécharge donc dans le dossier personnel.

Les commandes d'installation sont prévues pour Linux (Google Colab, ou WSL sous Windows).

In [ ]:
%pip install -q pyspark

In [ ]:
import os

HOME = os.path.expanduser("~")
os.environ["JAVA_HOME"] = HOME + "/jdk17"
os.environ["PATH"] = HOME + "/jdk17/bin:" + os.environ["PATH"]

if not os.path.exists(HOME + "/jdk17/bin/java"):
    !mkdir -p ~/jdk17
    !wget -qO- "https://api.adoptium.net/v3/binary/latest/17/ga/linux/x64/jdk/hotspot/normal/eclipse" | tar -xz -C ~/jdk17 --strip-components=1

!java -version

## 2. Fichiers de test

On reprend les fichiers de la partie 1. S'ils n'existent pas, on les recrée de la même façon.

In [ ]:
import random
import time

mots = ["chat", "chien", "arbre", "fleur", "maison", "soleil", "montagne", "livre",
        "ciel", "mer", "lune", "route", "oiseau", "jardin", "voiture"]

fichiers = {"mots_petit.txt": 1000, "mots_moyen.txt": 100000, "mots_grand.txt": 1000000}

for nom, n in fichiers.items():
    if not os.path.exists(nom):
        with open(nom, "w", encoding="utf-8") as f:
            for i in range(n):
                f.write(" ".join(random.choices(mots, k=10)) + "\n")

!ls -lh mots_*.txt

## 3. Démarrage de Spark

On lance Spark en mode local : `local[*]` veut dire que Spark utilise tous les cœurs du processeur comme s'ils étaient des machines d'un cluster.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("WordCount")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Version de Spark :", spark.version)
print("Nombre de cœurs utilisés :", sc.defaultParallelism)

## 4. WordCount avec PySpark

In [ ]:
def compter_spark(fichier):
    # une partition par cœur au minimum, pour que tous les cœurs travaillent
    lignes = sc.textFile(os.path.abspath(fichier), minPartitions=sc.defaultParallelism)
    mots = lignes.flatMap(lambda ligne: ligne.split())
    paires = mots.map(lambda mot: (mot, 1))
    comptes = paires.reduceByKey(lambda a, b: a + b)
    return dict(comptes.collect())


for mot, nb in sorted(compter_spark("mots_petit.txt").items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

On retrouve les étapes de MapReduce : `flatMap` découpe les lignes en mots, `map` crée les paires (mot, 1) et `reduceByKey` additionne les valeurs de chaque mot.

Les transformations (`flatMap`, `map`, `reduceByKey`) ne lancent aucun calcul : Spark construit seulement un plan. Le calcul ne démarre qu'avec une action, ici `collect()`, qui renvoie le résultat au programme Python.

Le fichier est découpé en partitions traitées en parallèle par les cœurs. `reduceByKey` additionne d'abord les comptes à l'intérieur de chaque partition avant de les échanger, comme le combiner de Hadoop, et les données intermédiaires restent en mémoire au lieu d'être écrites sur disque entre les étapes.

## 5. Tests selon la taille des fichiers

On compare Spark à la version Python simple de la partie 1, et on vérifie que les deux donnent le même résultat.

In [ ]:
def compter_occurrences(fichier):
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur


temps_python = {}
temps_spark = {}

for nom in fichiers:
    debut = time.time()
    resultat_python = compter_occurrences(nom)
    temps_python[nom] = time.time() - debut

    debut = time.time()
    resultat_spark = compter_spark(nom)
    temps_spark[nom] = time.time() - debut

    print(nom, ": même résultat =", resultat_python == resultat_spark)

In [ ]:
print("Fichier".ljust(16), "Python simple".rjust(14), "Spark".rjust(10))
for nom in fichiers:
    print(nom.ljust(16), f"{temps_python[nom]:12.3f} s", f"{temps_spark[nom]:8.3f} s")

Sur le petit fichier, Spark est beaucoup plus lent que Python simple. Pour chaque comptage, Spark doit découper le travail en tâches, les répartir sur les cœurs, et faire passer les données entre Python et la JVM de Spark. Ce coût fixe est énorme par rapport à quelques milliers de mots.

Quand le fichier grossit, ce coût fixe pèse de moins en moins et l'écart se réduit, car Spark traite les partitions en parallèle alors que la version Python n'utilise qu'un seul cœur.

Par rapport à Hadoop (autour de 40 secondes par job dans la partie 1), Spark est beaucoup plus rapide : il n'y a pas de job à soumettre à YARN ni de conteneurs à démarrer à chaque fois, et les résultats intermédiaires restent en mémoire au lieu de passer par le disque. C'est pour ça que Spark a largement remplacé MapReduce pour ce type de traitement.

## 6. Arrêt de Spark

In [ ]:
spark.stop()